# Demo 2 of 6 - Reliable ingestion of CSV, Avro and JSON

**Why we are doing this.** Kenny's complaint: large client files arrive in different formats, columns are added or reordered without warning, a bad file can break a load, and reprocessing creates duplicates.

**What we will prove.**
1. A file landing in cloud storage loads by itself, with no scheduler and no warehouse to manage (Snowpipe).
2. A large file split into parts loads in parallel.
3. A file with a new column and reordered columns loads without code changes (schema evolution).
4. A file with bad rows loads the good rows, and we can see exactly which rows were rejected and why.
5. Re-running the load does not create duplicates.
6. Avro and nested JSON load just as easily.

**Setting.** Amazon S3 stands in for Azure Blob Storage here. On Azure the same design uses Event Grid instead of S3 event notifications. The SQL is identical.

**How to read this notebook.** Every cell has one of four labels:
- **SETUP - ALREADY DONE (read only):** code that already exists in the account. It is shown so the engineers can see how it was built. You do not run it.
- **RUN DURING DEMO:** a SQL cell you run live. Click inside it and press the run button (or Cmd+Enter).
- **UI STEP:** something you do in the Snowsight screens or the AWS console, not in the notebook.
- **SAY THIS:** your talk track for that moment.

## How Snowpipe works (explain this before running anything)
1. A health plan drops a file into the S3 folder.
2. S3 sends an event notification ("a new file arrived") to a message queue owned by Snowflake.
3. The pipe hears the message and runs its saved COPY command on that one file.
4. The rows land in the Bronze table. The file itself stays in S3; Snowflake only reads it.

The **storage integration** gives Snowflake read permission on the bucket. The **stage** is just a named pointer to the folder. It stores nothing.

**For the Databricks engineers:**

| Concept | Databricks | Snowflake |
|---|---|---|
| Load files as they arrive | Auto Loader | Snowpipe with AUTO_INGEST=TRUE |
| Add new columns automatically | mergeSchema | ENABLE_SCHEMA_EVOLUTION=TRUE on the table |
| Capture bad records | badRecordsPath / rescued data column | ON_ERROR=CONTINUE plus VALIDATE_PIPE_LOAD |
| Remember which files were loaded | Checkpoint location | Built-in load metadata (14 days for pipes) |

### SETUP - ALREADY DONE (read only, do not run)
**Stage: a named pointer to the client drop folder in S3**

```sql
CREATE STAGE SOMATUS_DEMO.BRONZE.S3_CLIENT_DROP
  URL = 's3://mgupta-bucket/soybase/somatus/'
  STORAGE_INTEGRATION = S3_DATA_INTEGRATION     -- read permission, no keys in code
  DIRECTORY = (ENABLE = TRUE);
```

### SETUP - ALREADY DONE (read only, do not run)
**File formats: one per file type**

```sql
CREATE FILE FORMAT FF_CSV TYPE = CSV
  PARSE_HEADER = TRUE                     -- read column names from the header row
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  NULL_IF = ('')
  ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE  -- a file missing a column loads that column as NULL
  COMPRESSION = AUTO;                     -- .gz is detected automatically
CREATE FILE FORMAT FF_AVRO TYPE = AVRO;
CREATE FILE FORMAT FF_JSON TYPE = JSON STRIP_OUTER_ARRAY = TRUE;
```

### SETUP - ALREADY DONE (read only, do not run)
**Bronze table with schema evolution switched on**

```sql
CREATE TABLE SOMATUS_DEMO.BRONZE.MEMBERS_RAW (
  MEMBER_ID STRING, CLIENT_ID STRING, DOB DATE, GENDER STRING, CKD_STAGE STRING, STATE STRING,
  FILE_NAME STRING,            -- which file each row came from (lineage)
  LOADED_AT TIMESTAMP_LTZ      -- when it was loaded
)
ENABLE_SCHEMA_EVOLUTION = TRUE;  -- this setting lives on the TABLE, not on the pipe
```

### SETUP - ALREADY DONE (read only, do not run)
**The pipe: a saved COPY command plus an event listener. Serverless, no warehouse.**

```sql
CREATE PIPE SOMATUS_DEMO.BRONZE.PIPE_MEMBERS
  AUTO_INGEST = TRUE                                  -- load when S3 says a file arrived
AS
COPY INTO MEMBERS_RAW
FROM @S3_CLIENT_DROP/members/
FILE_FORMAT = (FORMAT_NAME = FF_CSV)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE               -- match file columns to table columns by NAME, ignoring upper/lower case.
                                                      -- Required for schema evolution. Other options: CASE_SENSITIVE, NONE (by position).
INCLUDE_METADATA = (FILE_NAME = METADATA$FILENAME, LOADED_AT = METADATA$START_SCAN_TIME)
ON_ERROR = CONTINUE;                                  -- reject only the bad rows, load the rest

-- PIPE_CLAIMS is the same pattern for Avro into CLAIMS_RAW.
-- PIPE_LABS loads each JSON record whole into a VARIANT column RAW in LABS_RAW.
```

> **SAY THIS:** Four settings carry this whole demo: auto ingest, match by column name, on error continue, and schema evolution on the table.

### RUN DURING DEMO
Set the role, warehouse and schema. Run this first, every time you open the notebook.

In [ ]:
%%sql -r ctx
USE ROLE ACCOUNTADMIN;
USE DATABASE SOMATUS_DEMO;
USE WAREHOUSE WH_INGEST;
USE SCHEMA BRONZE;

### RUN DURING DEMO
Show the three pipes. Scroll right to **notification_channel**: that is the Snowflake-owned queue S3 sends its messages to.

In [ ]:
%%sql -r pipes
SHOW PIPES IN SCHEMA SOMATUS_DEMO.BRONZE;

### RUN DURING DEMO
Show that all three Bronze tables are empty before we start. (If they are not, run the 99_Reset notebook first.)

In [ ]:
%%sql -r empty
SELECT 'MEMBERS_RAW' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM MEMBERS_RAW UNION ALL
SELECT 'CLAIMS_RAW', COUNT(*) FROM CLAIMS_RAW UNION ALL
SELECT 'LABS_RAW',   COUNT(*) FROM LABS_RAW;

## Step 1 - A large file, split into parts, loads by itself

### UI STEP - do this in the screen, not in the notebook
**Drop the four HUMANA files into S3**

1. Open the AWS console in another browser tab and go to **S3**.
2. Click bucket **mgupta-bucket**, then folder **soybase**, then **somatus**, then **members**.
3. Click **Upload**, then **Add files**.
4. On your laptop, open the folder **Accounts/Somatus/somatus-demo/files** and select the four files **members_HUMANA_DEMO_20261021_part1.csv.gz** through **part4.csv.gz** (about 5 million rows in total).
5. Click **Upload** at the bottom. Wait for the green success banner.
6. Switch back to the notebook and wait about 30 to 60 seconds before running the next cell.

### RUN DURING DEMO
Check the pipe. **executionState** should be RUNNING. **pendingFileCount** shows files still being loaded.

In [ ]:
%%sql -r pipe_status
SELECT SYSTEM$PIPE_STATUS('PIPE_MEMBERS');

### RUN DURING DEMO
Show each file's load result. The four parts load separately and in parallel. Re-run this cell during the later steps too.

In [ ]:
%%sql -r copy_history
SELECT FILE_NAME, STATUS, ROW_PARSED AS ROWS_IN_FILE, ROW_COUNT AS ROWS_LOADED,
       ERROR_COUNT AS ROWS_REJECTED, FIRST_ERROR_MESSAGE, LAST_LOAD_TIME
FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(TABLE_NAME => 'MEMBERS_RAW',
           START_TIME => DATEADD('hour', -2, CURRENT_TIMESTAMP())))
ORDER BY LAST_LOAD_TIME DESC;

### UI STEP - do this in the screen, not in the notebook
**Show the same load history in the screen**

1. In the left menu, click **Data**, then **Databases**.
2. Open **SOMATUS_DEMO**, then **BRONZE**, then **Pipes**, then click **PIPE_MEMBERS**.
3. Click the **Copy history** tab. Each file appears with its status and row count.

> **SAY THIS:** Nobody scheduled that and no warehouse was started. The file arrived and the pipe loaded it. You pay per file for the serverless compute.

## Step 2 - A schema change: new column plus reordered columns

### RUN DURING DEMO
Show the table's columns before the change. Note there is no DIALYSIS_MODALITY column.

In [ ]:
%%sql -r cols_before
SELECT COLUMN_NAME, DATA_TYPE FROM SOMATUS_DEMO.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'BRONZE' AND TABLE_NAME = 'MEMBERS_RAW' ORDER BY ORDINAL_POSITION;

### UI STEP - do this in the screen, not in the notebook
**Drop the AETNA file with the changed layout**

1. In the S3 console, go to **mgupta-bucket > soybase > somatus > members** again.
2. Click **Upload > Add files** and select **members_AETNA_DEMO_20261022_v2.csv.gz**. In this file the columns are in a different order and there is a new column, DIALYSIS_MODALITY.
3. Click **Upload**, then wait about 30 to 60 seconds.

### RUN DURING DEMO
Show the columns again. DIALYSIS_MODALITY now exists. Nobody ran ALTER TABLE.

In [ ]:
%%sql -r cols_after
SELECT COLUMN_NAME, DATA_TYPE FROM SOMATUS_DEMO.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'BRONZE' AND TABLE_NAME = 'MEMBERS_RAW' ORDER BY ORDINAL_POSITION;

### RUN DURING DEMO
Show the new column has values for the AETNA rows and NULL for the older HUMANA rows.

In [ ]:
%%sql -r new_col
SELECT DIALYSIS_MODALITY, COUNT(*) AS MEMBERS FROM MEMBERS_RAW GROUP BY 1 ORDER BY 2 DESC;

> **SAY THIS:** The reordered columns loaded correctly because we match by name, not by position. The new column was added because schema evolution is on. No pipeline was broken and no code was changed.

## Step 3 - A bad file: good rows load, bad rows are reported

### UI STEP - do this in the screen, not in the notebook
**Drop the CIGNA file that contains bad dates**

1. In the S3 console, go to **mgupta-bucket > soybase > somatus > members**.
2. Click **Upload > Add files** and select **members_CIGNA_DEMO_20261022_bad.csv**. It has 100 rows; 10 of them have the text 'not-a-date' in the date of birth column.
3. Click **Upload**, then wait about 30 to 60 seconds.
4. Re-run the **copy_history** cell above. The CIGNA file shows **PARTIALLY_LOADED** with 100 rows in the file, 90 loaded and 10 rejected.
5. In the screen, the same appears in **PIPE_MEMBERS > Copy history**. Hover over the small information icon next to 'Partially loaded' to see the first error message. The screen shows only a summary, so for all rejected rows use the next cell.

### RUN DURING DEMO
List every rejected row: the file, the line number, the error, the column and the full original record. This is what you send back to the health plan.

In [ ]:
%%sql -r rejected_rows
SELECT FILE, LINE, ERROR, COLUMN_NAME, REJECTED_RECORD
FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.VALIDATE_PIPE_LOAD(
       PIPE_NAME  => 'SOMATUS_DEMO.BRONZE.PIPE_MEMBERS',
       START_TIME => DATEADD('hour', -2, CURRENT_TIMESTAMP())));

> **SAY THIS:** One bad value does not stop the load. The 90 good rows are already usable, and you have an exact list of the 10 bad lines to send back to the client. When they resend a corrected file, it loads, and the Silver layer keeps only the latest record per member.

## Step 4 - Re-run the load: no duplicates

### RUN DURING DEMO
Note this row count.

In [ ]:
%%sql -r count_before
SELECT COUNT(*) AS ROWS_BEFORE FROM MEMBERS_RAW;

### RUN DURING DEMO
Ask the pipe to re-scan every file in the members folder, as an engineer would after an outage.

In [ ]:
%%sql -r refresh
ALTER PIPE PIPE_MEMBERS REFRESH;

### RUN DURING DEMO
Run this after about 30 seconds. The count is unchanged: the pipe remembers which files it already loaded and skips them.

In [ ]:
%%sql -r count_after
SELECT COUNT(*) AS ROWS_AFTER FROM MEMBERS_RAW;

> **SAY THIS:** Reprocessing is safe by default. A corrected file with new content is a new file and does load. Forcing a full reload is a deliberate choice (FORCE = TRUE), never an accident.

## Step 5 - Avro claims and nested JSON labs

### UI STEP - do this in the screen, not in the notebook
**Drop the claims and labs files**

1. In the S3 console, go to **mgupta-bucket > soybase > somatus**. If the folders **claims** and **labs** do not exist, click **Create folder** and create each one.
2. Open **claims**, click **Upload > Add files**, and select **claims_20261021_v1.avro** and **claims_20261022_v2.avro**. The v2 file has an extra field, PLACE_OF_SERVICE. Click **Upload**.
3. Open **labs**, click **Upload > Add files**, select **labs_20261021.json**, and click **Upload**. This file is needed later for Demo 5.
4. Wait about 60 seconds.

### RUN DURING DEMO
Show the Avro claims loaded, and that the new v2 field was added automatically, the same as for CSV.

In [ ]:
%%sql -r claims
SELECT COUNT(*) AS CLAIMS, COUNT(PLACE_OF_SERVICE) AS ROWS_WITH_NEW_V2_FIELD FROM CLAIMS_RAW;

### RUN DURING DEMO
Show the lab JSON stored whole in one VARIANT column. Click a cell to see the nested structure.

In [ ]:
%%sql -r labs_raw
SELECT RAW FROM LABS_RAW LIMIT 3;

### RUN DURING DEMO
Query inside the JSON with plain dot notation. No parsing code and no fixed schema needed at load time.

In [ ]:
%%sql -r labs_paths
SELECT RAW:member_id::STRING             AS MEMBER_ID,
       RAW:panel.results[0].test::STRING  AS FIRST_TEST,
       RAW:panel.results[0].value::FLOAT  AS FIRST_VALUE
FROM LABS_RAW LIMIT 5;

> **SAY THIS:** CSV, Avro and JSON all land the same way. Nested JSON is queryable with SQL immediately.